# 04 — Market Distribution

Measure return distribution, skewness, kurtosis, quantiles and tails.


In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Distribution Modules
import pandas as pd

from src.data.loaders import read_parquet, save_parquet
from src.features.returns import add_returns
from src.features.distribution import add_rolling_distribution_stats


In [3]:
# Cell: Build Distribution Features
clean = read_parquet("data/processed/clean_daily_ohlcv.parquet")
features = add_returns(clean)
features = add_rolling_distribution_stats(features, return_col="return_1d", window=60)
features.shape


(717835, 25)

In [4]:
# Cell: Summarize Distribution By Symbol
summary = (
    features.groupby("symbol").agg(
        mean_return=("return_1d", "mean"),
        median_return=("return_1d", "median"),
        skew_60d=("dist_skew_60d", "last"),
        kurt_60d=("dist_kurt_60d", "last"),
        tail_ratio_60d=("tail_ratio_60d", "last"),
    )
    .sort_values("tail_ratio_60d", ascending=False)
)
summary


,mean_return,median_return,skew_60d,kurt_60d,tail_ratio_60d
symbol,,,,,
KDC,0.000757,0.000000,-0.064330,4.962698,2.712263
ABW,0.000368,0.000000,0.845073,1.098798,2.621430
STB,0.001514,0.001493,1.388469,2.202504,2.596302
HII,0.000361,0.000000,0.967175,1.715604,2.399375
EVS,0.000689,0.000000,1.253682,2.423195,2.353037
...,...,...,...,...,...
VPG,-0.000497,0.000000,0.387527,1.888694,0.346007
THG,0.000487,0.000000,-0.933415,1.780990,0.270500
GIL,0.000407,0.000000,-1.425709,2.282923,0.191087


In [5]:
# Cell: Save Distribution Outputs
# Notebook-only scratch output -- not read by any production script (confirmed
# via grep, docs/reports/codebase_audit/MARKET_REGIME_AUDIT.md NV1). Kept out of
# data/processed/ on purpose so re-running this notebook by hand can never
# collide with scripts/generate_features.py's production output (see NV1 fix
# for 05/06_market_regime.ipynb in the same report).
save_parquet(features, "notebooks/outputs/stock_features_distribution.parquet")
features[["symbol", "date", "return_1d", "dist_skew_60d", "dist_kurt_60d", "tail_ratio_60d"]].tail(20)


,symbol,date,return_1d,dist_skew_60d,dist_kurt_60d,tail_ratio_60d
717815,YEG,2026-06-23,0.041860,0.921228,1.060779,1.396663
717816,YEG,2026-06-24,-0.003348,0.953717,1.118418,1.396663
717817,YEG,2026-06-25,0.069429,1.630687,4.264391,1.765252
717818,YEG,2026-06-26,0.013613,1.590342,4.080465,1.765252
717819,YEG,2026-06-29,-0.039256,1.355158,3.827784,1.564019
717820,YEG,2026-06-30,0.023656,1.259071,3.426377,1.564019
717821,YEG,2026-07-01,-0.009454,1.261162,3.405038,1.564019
717822,YEG,2026-07-02,-0.011665,1.267177,3.448161,1.564019
717823,YEG,2026-07-03,-0.010730,1.298261,3.460498,1.564019
717824,YEG,2026-07-06,-0.020607,1.380464,3.870642,1.564019
